Start with 3 features max:

- substation_count
- substation_density (count / county area)
- (optional later) distance_to_nearest_substation

Transmission lines can come later.

In [3]:
# pip install osmnx geopandas

Getting geometry from  census gov

In [4]:
import geopandas as gpd

# US counties (Census TIGER)
counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2020/COUNTY/tl_2020_us_county.zip")

counties["CountyFIPS"] = counties["STATEFP"] + counties["COUNTYFP"]

In [5]:
counties.head()

,STATEFP,COUNTYFP,COUNTYNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,CountyFIPS
0,31,039,00835841,31039,Cuming,Cuming County,06,H1,G4020,NaN,NaN,NaN,A,1477645345,10690204,+41.9158651,-096.7885168,"POLYGON ((-97.01952 42.0041, -97.01952 42.0049...",31039
1,53,069,01513275,53069,Wahkiakum,Wahkiakum County,06,H1,G4020,NaN,NaN,NaN,A,680976231,61568965,+46.2946377,-123.4244583,"POLYGON ((-123.43639 46.2382, -123.44759 46.24...",53069
2,35,011,00933054,35011,De Baca,De Baca County,06,H1,G4020,NaN,NaN,NaN,A,6016818946,29090018,+34.3592729,-104.3686961,"POLYGON ((-104.56739 33.99757, -104.56772 33.9...",35011
3,31,109,00835876,31109,Lancaster,Lancaster County,06,H1,G4020,339,30700,NaN,A,2169272970,22847034,+40.7835474,-096.6886584,"POLYGON ((-96.91075 40.78494, -96.91075 40.790...",31109
4,31,129,00835886,31129,Nuckolls,Nuckolls County,06,H1,G4020,NaN,NaN,NaN,A,1489645188,1718484,+40.1764918,-098.0468422,"POLYGON ((-98.27367 40.0894, -98.27367 40.0894...",31129


In [ ]:
counties.memory_usage(index=True)/1024**2 # units of MB

Index         0.000126
STATEFP       0.030842
COUNTYFP      0.033926
COUNTYNS      0.049347
GEOID         0.040094
NAME          0.046401
NAMELSAD      0.068247
LSAD          0.030842
CLASSFP       0.030842
MTFCC         0.040094
CSAFP         0.028653
CBSAFP        0.034196
METDIVFP      0.025584
FUNCSTAT      0.027758
ALAND         0.024673
AWATER        0.024673
INTPTLAT      0.058599
INTPTLON      0.061684
geometry      0.024673
CountyFIPS    0.040094
dtype: float64

Now catch this up to speed with data needed for RF stuff (from v4):

In [15]:
import pandas as pd

rf_v4_path = r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_transients\event_level_dataset_rf_v4_gdp.csv"

event_df = pd.read_csv(rf_v4_path)

# Ensure correct types
event_df["CountyFIPS"] = event_df["CountyFIPS"].astype(str).str.zfill(5)
event_df["event_start"] = pd.to_datetime(event_df["event_start"])

print(event_df.shape)
print(event_df.columns)
print(event_df.head())

(90, 19)
Index(['event_id', 'storm', 'CountyFIPS', 't90', 't90_censored', 'max_gust',
       'mean_gust_7d', 'total_precip_7d', 'pressure_min_7d',
       'max_customers_tracked', 'county_pop', 'event_start',
       'min_dist_to_storm_km', 'matched_ibtracs_storm',
       'storm_max_wind_center_in_conus_mps', 'storm_max_wind_near_conus_mps',
       'year', 'county_gdp_real_2017_dollars',
       'county_gdp_real_2017_per_capita'],
      dtype='str')
                            event_id          storm CountyFIPS         t90  \
0  2017212N28275_2017-07-30 21:00:00  2017212N28275      12075    1.885185   
1  2017242N16333_2017-09-09 19:00:00  2017242N16333      12037  120.406155   
2  2017242N16333_2017-09-09 20:00:00  2017242N16333      12067  186.802564   
3  2017242N16333_2017-09-09 21:00:00  2017242N16333      12013   52.651220   
4  2017242N16333_2017-09-09 22:00:00  2017242N16333      12075  170.764203   

   t90_censored   max_gust  mean_gust_7d  total_precip_7d  pressure_min_7d  \
0 

In [18]:
import geopandas as gpd

counties = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/COUNTY/tl_2020_us_county.zip"
)

counties["CountyFIPS"] = counties["STATEFP"] + counties["COUNTYFP"]

counties = counties[counties["CountyFIPS"].isin(counties_needed)].copy()

print(counties.shape)

(64, 19)


In [19]:
print(counties["CountyFIPS"].nunique(), "counties in geometry")
print(event_df["CountyFIPS"].nunique(), "counties in event_df")

64 counties in geometry
64 counties in event_df


In [23]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import time

# --- Configure OSM (more stable endpoint + timeout) ---
ox.settings.overpass_endpoint = "https://overpass.kumi.systems/api/interpreter"
ox.settings.timeout = 180

tags = {
    "power": ["substation", "plant", "transformer"]
}
subs_list = []

for idx, row in counties.iterrows():
    fips = row["CountyFIPS"]
    poly = row.geometry

    try:
        subs = ox.features_from_polygon(poly, tags=tags)

        if len(subs) > 0:
            subs = subs.reset_index()
            subs = gpd.GeoDataFrame(subs, geometry="geometry", crs="EPSG:4326")

            # Convert everything to points
            subs["geometry"] = subs.geometry.centroid

            subs["CountyFIPS"] = fips
            subs_list.append(subs)

        print(f"[{idx}] Success: {fips} | {len(subs)} substations")

    except Exception as e:
        print(f"[{idx}] Failed: {fips} | {e}")

    # --- Rate limiting (avoid API blocks) ---
    time.sleep(1)

# --- Combine all results ---
subs_all = pd.concat(subs_list, ignore_index=True)

print("Total substations collected:", len(subs_all))

# --- Save immediately (important: avoids re-querying) ---
subs_all.to_file(
    r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_transients\substations.geojson",
    driver="GeoJSON"
)

print("Saved substations to GeoJSON")

C:\Users\teaching\AppData\Local\Temp\ipykernel_17536\833917881.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subs["geometry"] = subs.geometry.centroid


[71] Success: 12053 | 45 substations


C:\Users\teaching\AppData\Local\Temp\ipykernel_17536\833917881.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subs["geometry"] = subs.geometry.centroid


[72] Success: 12129 | 24 substations


C:\Users\teaching\AppData\Local\Temp\ipykernel_17536\833917881.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subs["geometry"] = subs.geometry.centroid


...

[3106] Success: 12045 | 9 substations


C:\Users\teaching\AppData\Local\Temp\ipykernel_17536\833917881.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subs["geometry"] = subs.geometry.centroid


[3118] Success: 12073 | 88 substations


C:\Users\teaching\AppData\Local\Temp\ipykernel_17536\833917881.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subs["geometry"] = subs.geometry.centroid


[3207] Success: 12015 | 46 substations
Total substations collected: 5146
Saved substations to GeoJSON


might be some code in cgpt that does the aggregation. take a look.